In [1]:
import pandas as pd
import os
import re
import warnings
from pathlib import Path

!date
!whoami
!pwd

Wed Nov  4 07:34:57 PST 2026
lutzes
/mnt/share/homes/lutzes/vivarium_research_alzheimers/results_tables


In [2]:
final_csvs_dir = Path("/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/results/final_csvs")

# Use the most recent dated folder written by the results tables
# notebooks. To use a different folder, set results_date by hand, for
# example results_date = "2026_10_30".
results_date = max(
    path.name for path in final_csvs_dir.iterdir()
    if path.is_dir() and re.fullmatch(r"\d{4}_\d{2}_\d{2}", path.name)
)
print(results_date)
intermediate_output_dir = final_csvs_dir / results_date / "draws"
final_output_dir = final_csvs_dir / results_date / "no_draws"
final_output_dir.mkdir(exist_ok=True)

# Find all intermediate CSV files except for medication_doses.csv
with os.scandir(intermediate_output_dir) as entries:
    csv_names = [entry.name for entry in entries
                 if entry.is_file() and entry.name.endswith(".csv")
                 # Exclude medication_doses.csv since we don't want to
                 # give it to client
                 and entry.name != "medication_doses.csv"]

# Loop through each CSV file, read it, drop draw columns, and write to
# final output directory
for filename in csv_names:
    df = pd.read_csv(intermediate_output_dir / filename)
    draw_cols = [col for col in df.columns if col.startswith("draw_")]
    df = df.drop(columns=draw_cols)
    if (final_output_dir / filename).exists():
        warnings.warn(f"Overwriting existing file {final_output_dir / filename}")
    df.to_csv(final_output_dir / filename, index=False)
    # # Verify final output -- doesn't work due to rounding floating
    # # point values to 12 decimal places when writing to CSV
    # temp = pd.read_csv(final_output_dir / filename)
    # assert temp.equals(df), f"Data mismatch in {filename}"

csv_names

2026_11_04


['dalys.csv',
 'deaths.csv',
 'population.csv',
 'csf_pet_tests.csv',
 'incidence.csv',
 'prevalence.csv',
 'bbbm_tests.csv',
 'medication.csv']

In [3]:
df

,Year,Location,Age,Sex,Disease Stage,Scenario,Measure,Metric,Mean,95% UI Lower,95% UI Upper
0,2025,Brazil,65_plus,Both,Preclinical AD,"Accelerated Testing, With Treatment",Medication Completion,Number,0.000000,0.000000,0.000000
1,2025,Brazil,65_plus,Female,Preclinical AD,"Accelerated Testing, With Treatment",Medication Completion,Number,0.000000,0.000000,0.000000
2,2025,Brazil,65_plus,Male,Preclinical AD,"Accelerated Testing, With Treatment",Medication Completion,Number,0.000000,0.000000,0.000000
3,2025,Brazil,65_plus,Both,Preclinical AD,"Accelerated Testing, No Treatment",Medication Completion,Number,0.000000,0.000000,0.000000
4,2025,Brazil,65_plus,Female,Preclinical AD,"Accelerated Testing, No Treatment",Medication Completion,Number,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
437755,2100,United States of America,75_to_79,Female,Susceptible,"Slow Testing, With Treatment",Medication Initiation,Rate,0.000294,0.000291,0.000296
437756,2100,United States of America,75_to_79,Male,Susceptible,"Slow Testing, With Treatment",Medication Initiation,Rate,0.000317,0.000314,0.000319
437757,2100,United States of America,75_to_79,Both,Susceptible,"Slow Testing, No Treatment",Medication Initiation,Rate,0.000000,0.000000,0.000000
437758,2100,United States of America,75_to_79,Female,Susceptible,"Slow Testing, No Treatment",Medication Initiation,Rate,0.000000,0.000000,0.000000
